# Prédire le QI à 3 ans à partir de la HRV néonatale

*Groupe A · Jade · PRI 2026 · cohorte AuBE (CHU Saint-Étienne)*

**Question** : les indices de variabilité cardiaque (HRV) enregistrés à la naissance aident-ils à prédire le QI mesuré vers 3 ans (WPPSI-III), au-delà des facteurs cliniques ? Dans le rapport de Samuel Fornet, aucun modèle ne faisait mieux que prédire le QI moyen (MAE ≈ 10,3, R² ≤ 0).

| Étape | Section |
|---|---|
| **Mardi** · un enregistrement par enfant, scores complets, fusion, nettoyage | 1 |
| **Mardi** · réduction des variables : corrélation > 0,95, variance quasi nulle, forêt aléatoire, PCA | 2 |
| **Mercredi** · Elastic Net, Random Forest, LightGBM, XGBoost, SVR | 3 |
| **Jeudi** · 10 × 5 validation croisée, QIT / QIV / QIP / NCL, MAE, MSE, RMSE, R², `model_comparison.csv` | 4 |
| **Pourquoi ça n'apporte rien**, et à partir de quand ça aurait apporté quelque chose | 4.5 |
| **Contrôle positif** : la HRV retrouve-t-elle la prématurité ? | 5 |
| **Classification** QIT < 90 | 6 |
| **Hypothèses** : HRV plus tardive (M6 à M24) et trajectoires | 7 |
| **Conseils du prof** · prédiction conforme, explication des variables | 8, 9 |
| **Vendredi** · résumé pour le tuteur (`SUMMARY.md`), vérification des données | 10, 11 |

**Lancer** : `RAPIDE = True` pour vérifier que tout tourne (1 répétition, quelques minutes), puis `RAPIDE = False` pour les résultats à rapporter (10 répétitions, environ 25 minutes). Le code technique est replié : cliquer dessus pour l'afficher.

> **Données de santé d'enfants.** Les données et tout ce qui en est tiré au niveau individuel restent dans `data/` (ignoré par git). Le filtre `nbstripout` vide les sorties du notebook à chaque commit. Ne jamais donner ces données à un outil d'IA ; les sections 4.4 et 8 montrent des enfants un par un.

In [ ]:
import hashlib
import re
import subprocess
import sys
import time
import unicodedata
import warnings
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from lightgbm import LGBMRegressor
from scipy.stats import rankdata
from xgboost import XGBRegressor

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV, LogisticRegressionCV
from sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score, mean_absolute_error, mean_squared_error,
                             r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, StandardScaler
from sklearn.svm import SVR

# Racine du dépôt : le premier dossier qui contient evaluate_model.py, en remontant depuis celui du notebook.
RACINE = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "evaluate_model.py").exists()), None)
if RACINE is None:
    raise FileNotFoundError(f"evaluate_model.py introuvable en remontant depuis {Path.cwd()} : "
                            "ouvrir le notebook depuis le dépôt FSIQ-from-ECG.")
sys.path.insert(0, str(RACINE))
from evaluate_model import bootstrap, evaluate, load_labels, norm_id, run_cv   # évaluation commune de l'équipe

In [ ]:
# ---------------------------------------------------------------- Réglages principaux
RAPIDE = True                       # True : vérification rapide (1 répétition) ; False : résultats à rapporter (10)
CIBLES = ["QIT", "QIV", "QIP", "NCL"]   # QIT = cible principale ; retirer les autres pour aller plus vite
T_REPETITIONS, K_PLIS, K_INTERNE = 10, 5, 5   # 10 × 5 plis pour évaluer, 5 plis internes pour régler les modèles
N_ITER = 4 if RAPIDE else 12        # combinaisons d'hyperparamètres essayées (Random Forest, LightGBM, XGBoost, SVR)

SEUIL_CORR = 0.95                   # une seule variable gardée par groupe corrélé au-delà de ce seuil
SEUIL_QUASI_CONSTANTE = 0.95        # variable retirée si une même valeur concerne 95 % des enfants
TOP_K = 30                          # variables gardées par la sélection « sél. RF »
VARIANCE_PCA = 0.90                 # part de variance gardée par la sélection « PCA 90 % »

SEUIL_QIT = 90                      # classification (section 6) : QIT < 90 contre ≥ 90 ; [PISTE] 85
PASSAGES = ["M0", "M6", "M12", "M18", "M24"]       # hypothèse 1 (section 7) : HRV mesurée plus tard
TRAJECTOIRES = [("M0", "M12"), ("M0", "M24")]      # hypothèse 2 (section 7) : évolution de la HRV entre deux âges
ALPHA = 0.10                        # prédiction conforme : intervalle à 1 − α = 90 %

REPETITIONS = [1] if RAPIDE else None           # répétitions de folds.csv utilisées (None = les 10)
N_REP = 1 if RAPIDE else T_REPETITIONS          # répétitions des analyses des sections 4.5 à 7
GRAINE, N_JOBS = 42, -1             # N_JOBS = -1 : tous les cœurs du processeur

In [ ]:
# ---------------------------------------------------------------- Fichiers
DOSSIER_DONNEES = RACINE / "data"
DOSSIER_PARTAGE = DOSSIER_DONNEES / "cleaned"   # cohorte, plis, prédictions, jeu nettoyé : local seulement
DOSSIER_RESULTATS = Path.cwd() / "results"      # model_comparison.csv, SUMMARY.md, graphique : chiffres agrégés
FICHIER_HRV_LIN, FICHIER_HRV_NL = "HRV_Lineaire_Full.csv", "HRV_NonLineaire_Full.csv"
FICHIER_WPPSI = "Résultats WPPSI III.csv"
REGENERER_PLIS = False              # True seulement pour remplacer des cohort.csv / folds.csv venus d'ailleurs

# ---------------------------------------------------------------- Colonnes (voir GroupA/dataset.md)
# Les noms sont comparés sans majuscules, accents ni espaces ; un accent illisible (« Num�ro ») vaut n'importe quelle lettre.
COL_ID_HRV, COL_ID_WPPSI, COL_PASSAGE = "Numéro", "N°", "Passage"
PASSAGE_RETENU = "M0"               # analyse principale : enregistrement à la naissance (parfois écrit « MO »)
LIGNES_ENTETE_WPPSI = 2             # « Note composite » au-dessus de « QIT » -> « Note composite | QIT »
COLS_SCORES = {e: f"Note composite | {e}" for e in ["QIV", "QIP", "QIT", "NCL"]}
COLS_CLINIQUES = ["Sexe", "Prématuré", "SFA", "Réa à la naissance", "Pathologie neurologique",
                  "Tabac", "Diabète", "HTA", "Maturation Pulm", "Dépression: prise d'IRS"]
# Jamais en variables d'entrée : identité, dates, horaires, durées, nombres de RR, scores WPPSI.
MOTIFS_EXCLUS = [r"numero", r"^n°", r"monogramme", r"ddn", r"filename", r"recorder", r"date", r"duration",
                 r"beginning", r"\bend\b", r"nb ?rr", r"analy[sz]ed", r"passage", r"sortie",
                 r"^(qit|qiv|qip|ncl|com|inf|dim|cub|aob)\b"]
MAX_MANQUANTS_COL, MAX_MANQUANTS_LIGNE = 0.30, 0.50
MIN_CAS_RARES = 5                   # covariable binaire retirée si une classe a moins de 5 enfants

# ---------------------------------------------------------------- Elastic Net
# sklearn : `alpha` = force de la pénalité (λ dans glmnet), `l1_ratio` = mélange L1/L2 (α dans glmnet).
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]
N_ALPHAS, EPS_ALPHA = 30, 1e-2      # 30 valeurs d'alpha, de alpha_max (aucune variable) à alpha_max / 100

# 1. Preprocessing

## 1.1 Lecture des fichiers

Trois exports Excel séparés par des tabulations. Les fichiers HRV ont deux lignes d'en-tête : les noms, puis une ligne d'unités (M/F, 0/1, ms, %…) qui est ignorée. Le fichier WPPSI a aussi deux lignes d'en-tête, réunies en « Bloc | Colonne ». Les cases vides peuvent être écrites `""`, `" "`, `-` ou `NaN` : toutes deviennent des valeurs manquantes.

In [ ]:
def normaliser_texte(s):
    """Minuscules, sans accents, espaces simplifiés : ' Numéro ' -> 'numero'."""
    s = unicodedata.normalize("NFKD", str(s))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().lower()


def cle(s):
    return re.sub(r"[\s_.\-]", "", normaliser_texte(s))


def meme_nom(colonne, nom):
    """Compare les noms ; un caractère illisible « � » dans le nom de colonne vaut n'importe quel caractère."""
    motif = "".join("." if ch == "\ufffd" else re.escape(ch) for ch in cle(colonne))
    return re.fullmatch(motif, cle(nom)) is not None


def trouver_colonne(df, *noms):
    """Vrai nom de la colonne qui correspond à l'un des noms (ou à la fin d'un nom « Bloc | Nom »), sinon None."""
    for nom in noms:
        exactes = [c for c in df.columns if meme_nom(c, nom)]
        if exactes:
            return exactes[0]
        fins = [c for c in df.columns if meme_nom(str(c).split("|")[-1], nom)]
        if len(fins) == 1:
            return fins[0]
    return None


def aplatir_entetes(colonnes):
    """Deux lignes d'en-tête (cellules fusionnées d'Excel) -> « Bloc | Sous-colonne »."""
    noms, bloc = [], ""
    for haut, bas in colonnes:
        haut = "" if str(haut).startswith("Unnamed") else str(haut).strip()
        bas = "" if str(bas).startswith("Unnamed") else str(bas).strip()
        bloc = haut or bloc        # une cellule fusionnée n'est remplie que sur sa première colonne
        noms.append(f"{bloc} | {bas}" if bloc and bas else bloc or bas)
    return noms


UNITES = {"-", "n", "ms", "ms^2", "ms/beat", "bpm", "%", "hz", "hh:mm:ss", "0/1", "m/f", "unit"}
VALEURS_VIDES = {"", "-", "--", "na", "n/a", "nan", "?", "#n/a", "#div/0!", "#valeur!", "#value!"}


def lire_csv(chemin, lignes_entete=1):
    brut = Path(chemin).read_bytes()
    try:
        encodage = "utf-8-sig"
        brut.decode(encodage)
    except UnicodeDecodeError:
        encodage = "cp1252"
    sep = max(["\t", ";", ","], key=brut.decode(encodage, errors="replace").splitlines()[0].count)
    df = pd.read_csv(chemin, sep=sep, encoding=encodage, header=list(range(lignes_entete)),
                     decimal="," if sep != "," else ".")
    if lignes_entete > 1:
        df.columns = aplatir_entetes(df.columns)
    noms, vus = [], {}
    for c in df.columns:                       # colonnes sans nom (tabulations en fin de ligne) ou en double
        nom = str(c).strip() or "sans_nom"
        vus[nom] = vus.get(nom, 0) + 1
        noms.append(nom if vus[nom] == 1 else f"{nom}.{vus[nom] - 1}")
    df.columns = noms
    valeurs = [normaliser_texte(v) for v in df.iloc[0] if pd.notna(v) and str(v).strip()]
    unites = bool(valeurs) and sum(v in UNITES for v in valeurs) >= 0.5 * len(valeurs)
    if unites:                                 # ligne d'unités sous l'en-tête des fichiers HRV
        df = df.iloc[1:].reset_index(drop=True)
    for c in df.columns:                       # « - », espaces, virgules décimales : en nombres si 90 % s'y prêtent
        if not pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any():
            texte = df[c].astype("string").str.replace(r"[\s\u00a0%]", "", regex=True).str.replace(",", ".")
            texte = texte.where(~texte.str.lower().isin(VALEURS_VIDES).fillna(False).astype(bool))
            nombres = pd.to_numeric(texte, errors="coerce").astype("float64")
            if nombres.notna().sum() >= 0.9 * texte.notna().sum():
                df[c] = nombres
    print(f"{Path(chemin).name} : {df.shape[0]} lignes, {df.shape[1]} colonnes"
          + (", ligne d'unités ignorée" if unites else ""))
    return df


hrv_lin_brut = lire_csv(DOSSIER_DONNEES / FICHIER_HRV_LIN)
hrv_nl_brut = lire_csv(DOSSIER_DONNEES / FICHIER_HRV_NL)
wppsi_brut = lire_csv(DOSSIER_DONNEES / FICHIER_WPPSI, LIGNES_ENTETE_WPPSI)

## 1.2 Identifiants

Les fichiers sont reliés par le **numéro** du sujet (1re colonne), harmonisé comme dans `evaluate_model.py` (« 012 », « 12.0 » → « 12 »). `dataset.md` proposait Monogramme + DDN, mais des initiales se répètent d'un enfant à l'autre (voir `data/pulisci_righe.py`) ; la date de naissance sert donc de vérification (1.4).

In [ ]:
col_id = {"lin": trouver_colonne(hrv_lin_brut, COL_ID_HRV), "nl": trouver_colonne(hrv_nl_brut, COL_ID_HRV),
          "wppsi": trouver_colonne(wppsi_brut, COL_ID_WPPSI)}
if None in col_id.values():
    raise KeyError(f"Colonne identifiant introuvable : {col_id}. Corriger COL_ID_HRV / COL_ID_WPPSI.")
hrv_lin_brut = hrv_lin_brut.assign(id_enfant=hrv_lin_brut[col_id["lin"]].map(norm_id))
hrv_nl_brut = hrv_nl_brut.assign(id_enfant=hrv_nl_brut[col_id["nl"]].map(norm_id))
wppsi_brut = wppsi_brut.assign(id_enfant=wppsi_brut[col_id["wppsi"]].map(norm_id))
communs = set(hrv_lin_brut["id_enfant"].dropna()) & set(hrv_nl_brut["id_enfant"].dropna()) & set(wppsi_brut["id_enfant"].dropna())
print(f"{len(communs)} sujets présents dans les 3 fichiers (tous passages confondus)")

## 1.3 Scores WPPSI

| Score | Ce qu'il mesure |
|---|---|
| **QIT** | quotient intellectuel total (cible principale) |
| **QIV** | quotient verbal : langage, connaissances |
| **QIP** | quotient de performance : raisonnement non verbal, visuo-spatial |
| **NCL** | note composite de langage : vocabulaire |

Moyenne 100, écart-type 15 dans la population. Ces scores ne servent jamais de variables d'entrée. Comme dans `data/pulisci_righe.py`, un enfant sans tous ses scores est retiré (perdu de vue).

In [ ]:
wppsi = wppsi_brut.dropna(subset=["id_enfant"])
cols_scores = {e: trouver_colonne(wppsi, nom) for e, nom in COLS_SCORES.items()}
cols_scores = {e: c for e, c in cols_scores.items() if c is not None and pd.api.types.is_numeric_dtype(wppsi[c])}
if "QIT" not in cols_scores:
    raise ValueError("QIT introuvable : vérifier COLS_SCORES et LIGNES_ENTETE_WPPSI.")
CIBLES = [c for c in CIBLES if c in cols_scores]

scores = wppsi[["id_enfant", *cols_scores.values()]].rename(columns={c: e for e, c in cols_scores.items()})
n_qit = scores.dropna(subset=["QIT"]).groupby("id_enfant")["QIT"].nunique()
scores = scores[~scores["id_enfant"].isin(n_qit.index[n_qit > 1])].groupby("id_enfant").first()
incomplets = scores.isna().any(axis=1)
scores = scores[~incomplets & scores.apply(lambda s: s.between(40, 160)).all(axis=1)]
wppsi_enfant = wppsi.groupby("id_enfant").first()
print(f"{incomplets.sum()} sujets sans tous les scores retirés ; {len(scores)} enfants avec leurs scores")
display(scores.describe().loc[["mean", "std"]].round(1))

## 1.4 Un jeu de données par passage

Pour un passage donné (M0 = naissance, puis M6, M12, M18, M24), on garde un enregistrement par enfant, les indices HRV des deux fichiers, les covariables cliniques et les scores. On retire ensuite :
- les enfants absents d'un des 3 fichiers, ou dont la date de naissance diffère d'un fichier à l'autre ;
- les colonnes vides à ce passage : à M0, le fichier linéaire n'a que ses indices **24h**, ses colonnes Day et Night sont vides ;
- les variables manquantes pour plus de 30 % des enfants, ou quasi constantes (une même valeur pour 95 % des enfants) ;
- les covariables cliniques trop rares (moins de 5 enfants dans une classe).

Aucune de ces étapes ne regarde le QI : aucune information sur la cible ne fuit. Le détail est affiché pour M0, l'analyse principale ; les autres passages servent aux hypothèses de la section 7.

In [ ]:
def normaliser_passage(valeur):
    """'MO' (lettre O), 'm0', 'M00' -> 'M0' ; 'M06' -> 'M6'."""
    s = re.sub(r"\s", "", str(valeur)).upper()
    return re.sub(r"^M0+(?=\d)", "M", "M0" if s == "MO" else s)


def periode(colonne):
    """'SDNN (TDNight)' -> 'Night', 'Higuchi (24h)' -> '24h', sinon 'autre'."""
    m = re.search(r"(24h|Day|Night)\)$", colonne.split("|", 1)[-1])
    return m.group(1) if m else "autre"


def par_periode(colonnes):
    return pd.Series([periode(c) for c in colonnes], dtype=object).value_counts().to_dict()


def preparer_hrv(brut, bloc, passage, bavard):
    df = brut.dropna(subset=["id_enfant"])
    df = df[df[trouver_colonne(df, COL_PASSAGE)].map(normaliser_passage) == normaliser_passage(passage)]
    df = df.drop_duplicates("id_enfant").set_index("id_enfant")
    cliniques = [c for c in (trouver_colonne(df, n) for n in COLS_CLINIQUES) if c is not None]
    exclues = [c for c in df.columns
               if c in cliniques or c == col_id[bloc] or any(re.search(m, normaliser_texte(c)) for m in MOTIFS_EXCLUS)]
    vides = [c for c in df.columns if c not in exclues and df[c].isna().all()]
    indices = [c for c in df.columns if c not in exclues + vides and pd.api.types.is_numeric_dtype(df[c])]
    if bavard:
        print(f"[{bloc}] {len(df)} enregistrements à {passage} : {len(indices)} indices HRV {par_periode(indices)}, "
              f"{len(vides)} colonnes vides à ce passage {par_periode(vides)}")
    return df[indices].add_prefix(f"{bloc}|"), df.drop(columns=indices)


def dates_naissance(source, index):
    col = trouver_colonne(source, "DDN")
    if col is None:
        return None
    texte = source[col].astype("string").str.strip()
    return pd.to_datetime(texte, dayfirst=True, errors="coerce", format="mixed").reindex(index)


def construire_donnees(passage, bavard=False):
    """Indices HRV (linéaires + non linéaires), covariables cliniques et scores des enfants enregistrés à ce passage."""
    hrv_lin, meta_lin = preparer_hrv(hrv_lin_brut, "lin", passage, bavard)
    hrv_nl, meta_nl = preparer_hrv(hrv_nl_brut, "nl", passage, bavard)
    donnees = hrv_lin.join(hrv_nl, how="inner").join(scores, how="inner")
    n_fusion = len(donnees)

    ddn = pd.DataFrame({nom: d for nom, source in [("lin", meta_lin), ("nl", meta_nl), ("wppsi", wppsi_enfant)]
                        if (d := dates_naissance(source, donnees.index)) is not None})
    ddn_differentes = (ddn.notna().sum(axis=1) >= 2) & (ddn.nunique(axis=1) > 1)
    donnees = donnees[~ddn_differentes]

    clinique = pd.DataFrame(index=donnees.index)
    for nom in COLS_CLINIQUES:
        source = next((s for s in [meta_lin, meta_nl, wppsi_enfant] if trouver_colonne(s, nom) is not None), None)
        if source is not None:
            valeurs = source[trouver_colonne(source, nom)].reindex(donnees.index)
            clinique[f"clin|{nom}"] = valeurs if pd.api.types.is_numeric_dtype(valeurs) \
                else valeurs.astype(object).where(valeurs.notna(), np.nan)
    rares = [c for c in clinique.columns
             if clinique[c].nunique() < 2 or (clinique[c].nunique() == 2 and clinique[c].value_counts().min() < MIN_CAS_RARES)]
    donnees = donnees.join(clinique.drop(columns=rares))

    cols_hrv = list(hrv_lin.columns) + list(hrv_nl.columns)
    donnees[cols_hrv] = donnees[cols_hrv].replace([np.inf, -np.inf], np.nan)
    manquant = donnees[cols_hrv].isna().mean()
    dominant = donnees[cols_hrv].apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 1.0)
    trop_vides = manquant.index[manquant > MAX_MANQUANTS_COL]
    quasi_constantes = dominant.index[dominant >= SEUIL_QUASI_CONSTANTE].difference(trop_vides)
    donnees = donnees.drop(columns=trop_vides.union(quasi_constantes))
    incomplets = donnees[[c for c in cols_hrv if c in donnees.columns]].isna().mean(axis=1) > MAX_MANQUANTS_LIGNE
    donnees = donnees[~incomplets]

    if bavard:
        print(f"{n_fusion} enfants présents dans les 3 fichiers à {passage} ; retirés : {ddn_differentes.sum()} "
              f"(date de naissance différente), {incomplets.sum()} (trop incomplets) -> {len(donnees)} enfants")
        print(f"Covariables cliniques : {clinique.shape[1] - len(rares)} gardées, trop rares {rares}")
        print(f"Variables HRV : {len(trop_vides)} trop vides {par_periode(trop_vides)}, "
              f"{len(quasi_constantes)} quasi constantes {par_periode(quasi_constantes)}, "
              f"{len(cols_hrv) - len(trop_vides) - len(quasi_constantes)} gardées")
    return donnees, meta_lin


donnees, meta_lin = construire_donnees(PASSAGE_RETENU, bavard=True)

## 1.5 Cohorte et plis communs

`evaluate_model.py` ne compare des modèles que s'ils ont prédit **les mêmes enfants avec les mêmes plis** : `cohort.csv` (enfants et scores) et `folds.csv` (10 répétitions de 5 plis stratifiés sur le QIT). En attendant le `prepare_data.py` de l'équipe, le notebook les crée, et les recrée lui-même si la liste des enfants change. Des fichiers fournis par l'équipe dans `data/cleaned/` sont utilisés tels quels.

In [ ]:
fichier_cohorte, fichier_plis = DOSSIER_PARTAGE / "cohort.csv", DOSSIER_PARTAGE / "folds.csv"
marque = DOSSIER_PARTAGE / "cohort_folds.notebook"     # empreinte des fichiers créés par ce notebook


def empreinte():
    return "".join(hashlib.sha256(f.read_bytes()).hexdigest() for f in (fichier_cohorte, fichier_plis))


existent = fichier_cohorte.exists() and fichier_plis.exists()
crees_ici = existent and marque.exists() and marque.read_text() == empreinte()
perimes = crees_ici and set(pd.read_csv(fichier_cohorte, dtype={"child_id": str})["child_id"].map(norm_id)) != set(donnees.index)
if REGENERER_PLIS or not existent or perimes:
    DOSSIER_PARTAGE.mkdir(parents=True, exist_ok=True)
    donnees[list(scores.columns)].rename_axis("child_id").reset_index().to_csv(fichier_cohorte, index=False)
    tranches = pd.qcut(donnees["QIT"], q=5, labels=False, duplicates="drop")
    plis = [pd.DataFrame({"child_id": donnees.index[te], "repeat": r, "fold": k})
            for r in range(1, T_REPETITIONS + 1)
            for k, (_, te) in enumerate(StratifiedKFold(K_PLIS, shuffle=True, random_state=GRAINE + r)
                                        .split(donnees, tranches), start=1)]
    pd.concat(plis).to_csv(fichier_plis, index=False)
    marque.write_text(empreinte())
    print(f"cohort.csv et folds.csv créés : {len(donnees)} enfants, {T_REPETITIONS} × {K_PLIS} plis")
else:
    print("cohort.csv et folds.csv existants utilisés" + ("" if crees_ici else " (venus de l'équipe ou d'une version précédente)"))

plis_partages = pd.read_csv(fichier_plis, dtype={"child_id": str})
plis_partages["child_id"] = plis_partages["child_id"].map(norm_id)
absentes = [c for c in CIBLES if c not in pd.read_csv(fichier_cohorte, nrows=0).columns]
if absentes:
    print(f"Cibles absentes de cohort.csv, retirées : {absentes} (REGENERER_PLIS = True pour les avoir)")
    CIBLES = [c for c in CIBLES if c not in absentes]
etiquettes = load_labels(fichier_cohorte, CIBLES)
sans_variables = set(plis_partages["child_id"]) - set(donnees.index)
if sans_variables:
    raise ValueError(f"{len(sans_variables)} enfants de la cohorte commune n'ont pas de variables ici : comparer les "
                     "nettoyages avec l'équipe, ou REGENERER_PLIS = True pour repartir de ce notebook.")
donnees = donnees.loc[sorted(plis_partages["child_id"].unique(), key=int)]
donnees.rename_axis("child_id").to_csv(DOSSIER_PARTAGE / "dataset_m0.csv")

X = donnees.drop(columns=list(scores.columns))
cols_lin = [c for c in X.columns if c.startswith("lin|")]
cols_nl = [c for c in X.columns if c.startswith("nl|")]
cols_clin = [c for c in X.columns if c.startswith("clin|")]
CLIN_TEXTE = {c for c in cols_clin if not pd.api.types.is_numeric_dtype(X[c])}
print(f"Jeu final : {len(X)} enfants ; {len(cols_lin)} indices linéaires, {len(cols_nl)} non linéaires, "
      f"{len(cols_clin)} covariables cliniques ({', '.join(c[5:] for c in cols_clin)})")

## 1.6 Y a-t-il un signal dans les variables ?

Avant tout modèle, un repère simple : la corrélation de chaque variable avec le QIT. Avec beaucoup de variables, certaines sont corrélées au QIT **par pur hasard** ; on compare donc la plus forte corrélation observée à celle obtenue en mélangant les QIT entre enfants. Si les deux sont proches, aucune variable n'est liée au QIT, même prise seule, et aucun modèle ne pourra faire de miracle.

C'est descriptif : choisir des variables à partir de ces corrélations, hors des plis, serait une fuite d'information.

In [ ]:
numeriques = X.select_dtypes("number")
rho = numeriques.corrwith(donnees["QIT"], method="spearman").abs().sort_values(ascending=False)
melange = np.random.default_rng(0)
maxima_hasard = [numeriques.corrwith(pd.Series(melange.permutation(donnees["QIT"].to_numpy()), index=X.index),
                                     method="spearman").abs().max() for _ in range(50)]
rho_hasard, seuil_hasard = np.mean(maxima_hasard), np.quantile(maxima_hasard, 0.95)
SIGNAL_UNIVARIE = rho.iloc[0] > seuil_hasard
print(f"Plus forte corrélation |Spearman| avec le QIT : {rho.iloc[0]:.2f}. Par pur hasard, avec {numeriques.shape[1]} "
      f"variables, on obtient en moyenne {rho_hasard:.2f}, et jusqu'à {seuil_hasard:.2f} une fois sur 20.")
print("Conclusion :", "une corrélation dépasse le hasard, un lien faible existe peut-être." if SIGNAL_UNIVARIE else
      "rien ne dépasse le hasard : aucune variable n'est liée au QIT, même prise seule.")
print("Les 5 plus fortes :", rho.head(5).round(2).to_dict())

col_appareil = trouver_colonne(meta_lin, "Recorder")
if col_appareil is not None:
    effectifs = meta_lin[col_appareil].reindex(donnees.index).value_counts()
    print(f"\nAppareils d'enregistrement : {effectifs.to_dict()} enfants"
          + (" ; presque tous avec le même appareil, donc pas d'effet d'appareil à craindre." if (effectifs >= 5).sum() < 2 else
             " ; [PISTE] comparer le QIT moyen par appareil (effet d'appareil possible)."))

# 2. Feature selection

Avec environ 70 variables HRV pour 189 enfants, un modèle peut apprendre le bruit des enfants d'entraînement au lieu d'un vrai lien. On réduit donc les variables **dans chaque pli, sur les seuls enfants d'entraînement** (sinon les enfants de test « aident » à préparer le modèle et les résultats sont trop optimistes) :

1. **Toujours** : remplir les trous par la médiane, garder une seule variable par groupe corrélé à plus de 0,95, transformer chaque variable vers une loi normale (les indices HRV sont très asymétriques).
2. **Au choix, selon l'expérience** :

| Option | Principe |
|---|---|
| `toutes` | pas de sélection : le modèle fait le tri lui-même (Elastic Net met des coefficients à zéro) |
| `sél. RF` | les 30 variables les plus importantes d'une forêt aléatoire de référence (conseil du prof) |
| `PCA 90 %` | les composantes principales qui gardent 90 % de la variance (conseil du prof) |

In [ ]:
class FiltreCorrelation(BaseEstimator, TransformerMixin):
    """Garde une variable par groupe de variables très corrélées (|Spearman| > seuil), ajusté dans chaque pli."""

    def __init__(self, seuil=0.95):
        self.seuil = seuil

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        if X.shape[1] < 2:
            self.garder_ = np.arange(X.shape[1])
            return self
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.abs(np.nan_to_num(np.corrcoef(rankdata(X, axis=0), rowvar=False)))
        garder = []
        for j in range(X.shape[1]):
            if not garder or corr[j, garder].max() <= self.seuil:
                garder.append(j)
        self.garder_ = np.array(garder)
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.garder_]

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = [f"x{i}" for i in range(self.n_features_in_)]
        return np.asarray(input_features, dtype=object)[self.garder_]


def faire_preprocesseur(colonnes):
    """Covariables cliniques (préfixe clin|) d'un côté, tout le reste (indices HRV) de l'autre."""
    hrv = [c for c in colonnes if not c.startswith("clin|")]
    clin_num = [c for c in colonnes if c.startswith("clin|") and c not in CLIN_TEXTE]
    clin_texte = [c for c in colonnes if c in CLIN_TEXTE]
    etapes = []
    if hrv:
        etapes.append(("hrv", Pipeline([("imputation", SimpleImputer(strategy="median")),
                                        ("correlation", FiltreCorrelation(SEUIL_CORR)),
                                        ("loi_normale", QuantileTransformer(n_quantiles=50, output_distribution="normal"))]),
                       hrv))
    if clin_num:
        etapes.append(("clin_num", Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                                            ("echelle", StandardScaler())]), clin_num))
    if clin_texte:
        etapes.append(("clin_texte", Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                                              ("codage", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                                                                       sparse_output=False)),
                                              ("echelle", StandardScaler())]), clin_texte))
    return ColumnTransformer(etapes, verbose_feature_names_out=False)


SELECTIONS = {
    "toutes": lambda: "passthrough",
    "sél. RF": lambda: SelectFromModel(RandomForestRegressor(n_estimators=200, max_features="sqrt", min_samples_leaf=5,
                                                             random_state=0, n_jobs=1),
                                       max_features=lambda X: min(TOP_K, X.shape[1]), threshold=-np.inf),
    "PCA 90 %": lambda: PCA(n_components=VARIANCE_PCA, svd_solver="full"),
}
JEUX = {"Clinique seul": cols_clin, "HRV seule": cols_lin + cols_nl, "Clinique + HRV": cols_clin + cols_lin + cols_nl}

# 3. Models

Cinq modèles de régression, réglés dans une **boucle interne à 5 plis** sur les seuls enfants d'entraînement :
- **Elastic Net** : régression linéaire pénalisée, la référence ; `ElasticNetCV` choisit la force de la pénalité ;
- **Random Forest, LightGBM, XGBoost** : arbres de décision, capables de liens non linéaires (le prof conseille LightGBM et XGBoost) ;
- **SVR** : régression à noyau.

Pour ces quatre derniers, `RandomizedSearchCV` essaie `N_ITER` combinaisons tirées dans les espaces d'hyperparamètres du plan du mercredi. Une **expérience** = un modèle + une sélection + un jeu de variables ; toutes voient les mêmes enfants et les mêmes plis. `Clinique seul` est la référence clinique : ce que la HRV devrait améliorer.

Un 6e modèle sert aux sections 5 et 6 : la **régression logistique**, pour prédire une classe (prématuré ou non, QIT < 90 ou non).

In [ ]:
INTERNE = KFold(K_INTERNE, shuffle=True, random_state=0)
ESPACES = {   # plan du mercredi
    "Random Forest": {"n_estimators": [200, 500], "max_depth": [3, 5, 7, None], "min_samples_leaf": [5, 10, 20],
                      "max_features": ["sqrt", "log2"]},
    "LightGBM": {"n_estimators": [100, 300], "learning_rate": [0.01, 0.05, 0.1], "max_depth": [3, 5, -1],
                 "min_child_samples": [10, 20], "reg_alpha": [0, 0.1, 1], "reg_lambda": [0, 1, 10]},
    "XGBoost": {"n_estimators": [100, 300], "learning_rate": [0.01, 0.05, 0.1], "max_depth": [3, 5, 7],
                "min_child_weight": [1, 5], "reg_alpha": [0, 0.1, 1], "reg_lambda": [1, 10], "subsample": [0.8, 1.0]},
    "SVR": {"C": [0.1, 1, 10, 100], "epsilon": [0.01, 0.1, 1], "kernel": ["rbf", "linear"], "gamma": ["scale", "auto"]},
}
BASES = {
    "Random Forest": lambda: RandomForestRegressor(random_state=0, n_jobs=1),
    "LightGBM": lambda: LGBMRegressor(colsample_bytree=0.5, subsample=0.8, subsample_freq=1, random_state=0,
                                      n_jobs=1, verbose=-1),
    "XGBoost": lambda: XGBRegressor(colsample_bytree=0.5, random_state=0, n_jobs=1),
    "SVR": lambda: SVR(),
}


def faire_modele(nom):
    if nom == "Elastic Net":
        return ElasticNetCV(l1_ratio=L1_RATIOS, alphas=N_ALPHAS, eps=EPS_ALPHA, cv=INTERNE, max_iter=10_000,
                            selection="random", random_state=0)
    return RandomizedSearchCV(BASES[nom](), ESPACES[nom], n_iter=N_ITER, cv=INTERNE,
                              scoring="neg_mean_absolute_error", random_state=0)


def faire_pipeline(modele, selection, colonnes):
    return Pipeline([("prep", faire_preprocesseur(colonnes)), ("selection", SELECTIONS[selection]()),
                     ("modele", faire_modele(modele))])


def faire_elastic_net(colonnes):
    return faire_pipeline("Elastic Net", "toutes", colonnes)


def faire_classifieur(colonnes):
    # Régression logistique pénalisée ; class_weight="balanced" : la classe rare compte autant que l'autre.
    return Pipeline([("prep", faire_preprocesseur(colonnes)),
                     ("modele", LogisticRegressionCV(Cs=10, cv=StratifiedKFold(3, shuffle=True, random_state=0),
                                                     scoring="roc_auc", class_weight="balanced", max_iter=5_000))])


EXPERIENCES = [                               # (modèle, sélection, jeu de variables)
    ("Elastic Net", "toutes", "Clinique seul"),
    ("Elastic Net", "toutes", "Clinique + HRV"),
    ("Random Forest", "toutes", "Clinique + HRV"),
    ("LightGBM", "sél. RF", "Clinique + HRV"),
    ("XGBoost", "sél. RF", "Clinique + HRV"),
    ("SVR", "PCA 90 %", "Clinique + HRV"),
]
EXPERIENCES = [e for e in EXPERIENCES if JEUX[e[2]]]

## 3.1 Vérification rapide

Chaque expérience est ajustée une fois, sur le 1er pli, avant la longue évaluation : une erreur de configuration apparaît ici en quelques secondes. La référence est calculée sur le même pli ; une seule mesure ne permet pas de conclure.

In [ ]:
def nom_experience(exp):
    return " · ".join(exp)


def taches_plis(repetitions=REPETITIONS):
    """(répétition, pli, enfants d'entraînement, enfants de test) des plis communs, découpés comme dans run_cv."""
    taches = []
    for r in sorted(plis_partages["repeat"].unique()):
        if repetitions is not None and r not in repetitions:
            continue
        fr = plis_partages[plis_partages["repeat"] == r]
        for k in sorted(fr["fold"].unique()):
            taches.append((r, k, fr.loc[fr["fold"] != k, "child_id"].tolist(), fr.loc[fr["fold"] == k, "child_id"].tolist()))
    return taches


def plis_internes(y, strates, n_rep=N_REP):
    """Plis propres à une analyse (autre cohorte, autre cible) : n_rep répétitions de K_PLIS plis stratifiés."""
    return [(r, k, list(y.index[tr]), list(y.index[te])) for r in range(1, n_rep + 1)
            for k, (tr, te) in enumerate(StratifiedKFold(K_PLIS, shuffle=True, random_state=GRAINE + r)
                                         .split(y, strates), start=1)]


def details_modele(pipe):
    """Hyperparamètres choisis et, pour Elastic Net, coefficients."""
    modele = pipe.named_steps["modele"]
    if isinstance(modele, RandomizedSearchCV):
        return {"hyper": modele.best_params_}
    if isinstance(modele, ElasticNetCV):
        return {"hyper": {"alpha": modele.alpha_, "l1_ratio": modele.l1_ratio_}, "coefs": modele.coef_}
    return {}


def ajuster_predire(exp, train_ids, test_ids, etiq):
    """Un pli de l'évaluation principale : apprend sur train_ids, prédit test_ids (un modèle par cible)."""
    warnings.filterwarnings("ignore")
    modele, selection, jeu = exp
    colonnes = JEUX[jeu]
    predictions, details = {}, {}
    for cible in etiq.columns:
        pipe = faire_pipeline(modele, selection, colonnes).fit(X.loc[train_ids, colonnes], etiq.loc[train_ids, cible])
        predictions[cible] = np.ravel(pipe.predict(X.loc[test_ids, colonnes]))
        details[cible] = details_modele(pipe)
    return predictions, details


def predictions_croisees(Xd, y, colonnes, fabrique, plis, proba=False):
    """Prédictions hors pli (une ligne par répétition, une colonne par enfant) pour les analyses des sections 4.5 à 7."""
    def un_pli(tr, te):
        warnings.filterwarnings("ignore")
        pipe = fabrique(colonnes).fit(Xd.loc[tr, colonnes], y.loc[tr])
        return pipe.predict_proba(Xd.loc[te, colonnes])[:, 1] if proba else np.ravel(pipe.predict(Xd.loc[te, colonnes]))
    sorties = Parallel(n_jobs=N_JOBS)(delayed(un_pli)(tr, te) for _, _, tr, te in plis)
    P = pd.DataFrame(np.nan, index=sorted({p[0] for p in plis}), columns=y.index)
    for (r, _, _, te), s in zip(plis, sorties):
        P.loc[r, te] = s
    return P


def gain_vs_moyenne(y, P, plis):
    """MAE du modèle, MAE de la moyenne d'entraînement de chaque pli, gain et son IC 95 % (bootstrap sur les enfants)."""
    B = pd.DataFrame(np.nan, index=P.index, columns=y.index)
    for r, _, tr, te in plis:
        B.loc[r, te] = y.loc[tr].mean()
    Y, Pm, Bm = y.to_numpy(float), P.to_numpy(float), B.to_numpy(float)
    tirages = bootstrap(Y, {"moyenne": Bm, "modele": Pm}, 2000, 0)
    gains = tirages["moyenne"]["MAE"] - tirages["modele"]["MAE"]
    mae_m, mae_b = np.abs(Pm - Y).mean(), np.abs(Bm - Y).mean()
    return {"enfants": len(y), "MAE moyenne": mae_b, "MAE modèle": mae_m, "gain": mae_b - mae_m,
            "IC bas": np.percentile(gains, 2.5), "IC haut": np.percentile(gains, 97.5)}


def resume_classification(y_bin, P):
    """AUC (moyenne des répétitions, IC 95 % par bootstrap sur les enfants) et performances au seuil 0,5."""
    yb = y_bin.to_numpy(int)
    proba = P.mean(axis=0).to_numpy(float)
    tirage = np.random.default_rng(0)
    aucs = []
    for _ in range(2000):
        i = tirage.integers(0, len(yb), len(yb))
        if 0 < yb[i].sum() < len(i):
            aucs.append(roc_auc_score(yb[i], proba[i]))
    classe = (proba >= 0.5).astype(int)
    return {"AUC": np.mean([roc_auc_score(yb, P.loc[r].to_numpy(float)) for r in P.index]),
            "AUC IC bas": np.percentile(aucs, 2.5), "AUC IC haut": np.percentile(aucs, 97.5),
            "sensibilité": recall_score(yb, classe), "spécificité": recall_score(yb, classe, pos_label=0),
            "exactitude": (classe == yb).mean(), "exactitude équilibrée": balanced_accuracy_score(yb, classe),
            "kappa": cohen_kappa_score(yb, classe)}


def essai(exp, tr, te):
    t = time.time()
    pred, _ = ajuster_predire(exp, tr, te, etiquettes[["QIT"]])
    return nom_experience(exp), mean_absolute_error(etiquettes.loc[te, "QIT"], pred["QIT"]), time.time() - t


_, _, tr0, te0 = taches_plis([1])[0]
essais = Parallel(n_jobs=N_JOBS)(delayed(essai)(exp, tr0, te0) for exp in EXPERIENCES)
reference_pli = mean_absolute_error(etiquettes.loc[te0, "QIT"], np.full(len(te0), etiquettes.loc[tr0, "QIT"].mean()))
display(pd.DataFrame([("QIT moyen (référence)", reference_pli, 0.0)] + essais,
                     columns=["expérience", "MAE (1er pli, QIT)", "secondes"]).set_index("expérience").round(2))

# 4. Evaluation

Protocole du jeudi : **10 répétitions × 5 plis** (`folds.csv`) ; dans chaque pli, le modèle apprend sur 4/5 des enfants et prédit le 5e. Les prédictions sont écrites au format de `evaluate_model.py` (`data/cleaned/predictions/`), donc comparables avec l'autre groupe. Avec `RAPIDE = True`, seule la 1re répétition est faite.

## 4.1 Lancement

In [ ]:
DOSSIER_PRED = DOSSIER_PARTAGE / "predictions"
DOSSIER_PRED.mkdir(parents=True, exist_ok=True)
REFERENCE = "score moyen (référence)"


def lancer(exp, fichier):
    """Ajuste l'expérience sur tous les plis en parallèle, puis écrit les prédictions avec run_cv (format commun)."""
    taches = taches_plis()
    sorties = Parallel(n_jobs=N_JOBS)(delayed(ajuster_predire)(exp, tr, te, etiquettes) for _, _, tr, te in taches)
    deja_calcule = {tuple(te): pred for (_, _, _, te), (pred, _) in zip(taches, sorties)}
    run_cv(lambda tr, te: deja_calcule[tuple(te)], fichier_plis, fichier_cohorte, fichier,
           targets=CIBLES, repeats=REPETITIONS, verbose=False)
    return [details for _, details in sorties]


fichiers, details_exp = {}, {}
debut = time.time()
for exp in EXPERIENCES:
    t = time.time()
    nom = nom_experience(exp)
    fichiers[nom] = DOSSIER_PRED / f"predictions_{re.sub(r'[^a-z0-9]+', '_', normaliser_texte(nom)).strip('_')}.csv"
    details_exp[nom] = lancer(exp, fichiers[nom])
    print(f"{nom} : {time.time() - t:.0f} s")
print(f"Total : {(time.time() - debut) / 60:.1f} min")

## 4.2 Comparaison des modèles

Une mesure par pli de test, résumée par moyenne ± écart-type (MAE, MSE, RMSE, R²) et écrite dans `results/model_comparison.csv`. La **référence** prédit, dans chaque pli, le score moyen des enfants d'entraînement : c'est le plancher à battre. Le graphique de gauche montre le **gain** de chaque modèle sur cette référence : à droite de 0, le modèle fait mieux que la moyenne.

In [ ]:
def mesures(vrai, pred):
    mse = mean_squared_error(vrai, pred)
    return {"MAE": mean_absolute_error(vrai, pred), "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(vrai, pred)}


lignes = []
for cible in CIBLES:
    for r, k, tr, te in taches_plis():
        lignes.append({"model": REFERENCE, "target": cible, "repeat": r, "fold": k,
                       **mesures(etiquettes.loc[te, cible], np.full(len(te), etiquettes.loc[tr, cible].mean()))})
    for nom, fichier in fichiers.items():
        p = pd.read_csv(fichier)
        lignes += [{"model": nom, "target": cible, "repeat": r, "fold": k, **mesures(g[f"true_{cible}"], g[f"predicted_{cible}"])}
                   for (r, k), g in p.groupby(["repeat", "fold"])]
par_pli = pd.DataFrame(lignes)
mae_reference = par_pli[par_pli.model == REFERENCE].set_index(["target", "repeat", "fold"])["MAE"]
par_pli["gain"] = mae_reference.reindex(pd.MultiIndex.from_frame(par_pli[["target", "repeat", "fold"]])).to_numpy() - par_pli["MAE"]

comparaison_modeles = par_pli.groupby(["model", "target"], sort=False)[["MAE", "MSE", "RMSE", "R2"]].agg(["mean", "std"])
comparaison_modeles.columns = [f"{m.lower()}_{s}" for m, s in comparaison_modeles.columns]
comparaison_modeles = comparaison_modeles.reset_index()
DOSSIER_RESULTATS.mkdir(parents=True, exist_ok=True)
comparaison_modeles.round(4).to_csv(DOSSIER_RESULTATS / "model_comparison.csv", index=False)

MEILLEURS = {c: comparaison_modeles[(comparaison_modeles.target == c) & (comparaison_modeles.model != REFERENCE)]
             .sort_values("mae_mean")["model"].iloc[0] for c in CIBLES}
tableau = comparaison_modeles.sort_values(["target", "mae_mean"], key=lambda s: s.map(CIBLES.index) if s.name == "target" else s)
display(pd.DataFrame({
    "MAE": [f"{m:.2f} ± {s:.2f}" for m, s in zip(tableau.mae_mean, tableau.mae_std)],
    "RMSE": [f"{m:.2f} ± {s:.2f}" for m, s in zip(tableau.rmse_mean, tableau.rmse_std)],
    "R²": [f"{m:.3f} ± {s:.3f}" for m, s in zip(tableau.r2_mean, tableau.r2_std)],
}, index=pd.MultiIndex.from_frame(tableau[["target", "model"]], names=["cible", "modèle"])))

modeles = list(fichiers)
gains = par_pli[par_pli.model != REFERENCE].groupby(["model", "target"])["gain"].agg(["mean", "std"])
fig, axes = plt.subplots(1, 2, figsize=(13, 0.5 * len(modeles) * max(1, len(CIBLES) / 2) + 1.5), sharey=True)
hauteur = 0.8 / len(CIBLES)
for j, cible in enumerate(CIBLES):
    t = comparaison_modeles[comparaison_modeles.target == cible].set_index("model").loc[modeles]
    g = gains.xs(cible, level="target").loc[modeles]
    pos = np.arange(len(modeles)) + (j - (len(CIBLES) - 1) / 2) * hauteur
    axes[0].barh(pos, g["mean"], height=hauteur, xerr=g["std"], label=cible, capsize=2)
    axes[1].barh(pos, t.r2_mean, height=hauteur, xerr=t.r2_std, label=cible, capsize=2)
for ax in axes:
    ax.axvline(0, color="k", lw=1)
axes[0].set(yticks=range(len(modeles)), yticklabels=modeles,
            xlabel="gain de MAE par rapport au score moyen (points) ; à droite de 0 = mieux")
axes[0].invert_yaxis()
axes[1].set(xlabel="R² ; à droite de 0 = mieux que la moyenne")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=len(CIBLES), frameon=False)
plt.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(DOSSIER_RESULTATS / "comparaison_modeles.png", dpi=150)
plt.show()

## 4.3 Intervalles de confiance (`evaluate_model.py`)

Le rapport commun de l'équipe, sur toutes les prédictions hors pli : MAE et R² avec un intervalle de confiance à 95 % (bootstrap sur les enfants), et le gain par rapport à la moyenne. **Si l'intervalle du gain contient 0, l'amélioration n'est pas démontrée.**

In [ ]:
resume_equipe, comparaison = evaluate(fichier_plis, fichier_cohorte, [str(f) for f in fichiers.values()],
                                      names=list(fichiers), targets=CIBLES, verbose=False)


def ic(bas, haut, chiffres=2):
    return f"[{bas:.{chiffres}f} ; {haut:.{chiffres}f}]"


r = resume_equipe
display(pd.DataFrame({
    "MAE [IC 95 %]": [f"{m:.2f} {ic(a, b)}" for m, a, b in zip(r.MAE, r.MAE_ci_low, r.MAE_ci_high)],
    "R² [IC 95 %]": [f"{m:.3f} {ic(a, b, 3)}" for m, a, b in zip(r.R2, r.R2_ci_low, r.R2_ci_high)],
    "gain vs moyenne [IC 95 %]": ["" if m == "mean_baseline" else f"{g:+.2f} {ic(a, b)}"
                                  for m, g, a, b in zip(r.model, r.MAE_gain_vs_baseline, r.gain_ci_low, r.gain_ci_high)],
    "gain démontré": ["" if m == "mean_baseline" else ("oui" if a > 0 else "non") for m, a in zip(r.model, r.gain_ci_low)],
}, index=pd.MultiIndex.from_frame(r[["target", "model"]].replace({"mean_baseline": REFERENCE}), names=["cible", "modèle"])))

## 4.4 Prédit contre observé (QIT, meilleur modèle)

Un point = un enfant, prédit quand il était dans le pli de test (1re répétition). **Graphique individuel : à ne pas partager.** Si les points forment une bande horizontale autour du QIT moyen, le modèle prédit presque la même chose pour tout le monde : il n'a pas trouvé de signal.

In [ ]:
MEILLEUR = MEILLEURS["QIT"]
exp_meilleur = next(e for e in EXPERIENCES if nom_experience(e) == MEILLEUR)
hors_pli = pd.read_csv(fichiers[MEILLEUR])
hors_pli = hors_pli[hors_pli["repeat"] == hors_pli["repeat"].min()]
vrai, pred = hors_pli["true_QIT"].to_numpy(), hors_pli["predicted_QIT"].to_numpy()
residus = vrai - pred

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
bornes = [min(vrai.min(), pred.min()) - 2, max(vrai.max(), pred.max()) + 2]
axes[0].scatter(vrai, pred, s=12, alpha=0.6)
axes[0].plot(bornes, bornes, "k--", lw=1, label="prédiction parfaite")
axes[0].axhline(vrai.mean(), color="grey", lw=1, label="QIT moyen")
axes[0].set(xlabel="QIT observé", ylabel="QIT prédit", title=MEILLEUR)
axes[0].legend()
axes[1].scatter(pred, residus, s=12, alpha=0.6)
axes[1].axhline(0, color="k", ls="--", lw=1)
axes[1].set(xlabel="QIT prédit", ylabel="résidu (observé − prédit)", title="Résidus")
plt.tight_layout()
plt.show()
print(f"Écart-type des prédictions : {pred.std():.1f} points (celui des vrais QIT : {vrai.std():.1f}). "
      f"MAE pour les QIT < {SEUIL_QIT} : {np.abs(residus[vrai < SEUIL_QIT]).mean():.1f} points, "
      f"pour les autres : {np.abs(residus[vrai >= SEUIL_QIT]).mean():.1f}.")

## 4.5 Pourquoi ça n'apporte rien, et quand ça aurait apporté quelque chose

**Pourquoi.** Trois raisons, de la plus importante à la moins importante :
1. **Il n'y a pas de lien dans les variables** (section 1.6) : aucune variable n'est plus corrélée au QIT que ce que donne le hasard. Un modèle ne crée pas un signal absent ; il ne peut que combiner ceux qui existent.
2. **Avec 189 enfants, chercher un signal qui n'existe pas coûte cher.** Chaque coefficient appris sur du bruit ajoute de l'erreur. Elastic Net le sent et met presque tout à zéro (il retombe sur la moyenne) ; les arbres, plus flexibles, apprennent le bruit et font **moins bien** que la moyenne (section 4.3).
3. **Le QI à 3 ans dépend de beaucoup de choses absentes ici** : l'environnement familial, la stimulation, le niveau socio-économique, les événements des trois premières années. La HRV à la naissance ne voit rien de tout cela.

**Quand ça aurait apporté quelque chose.** Pour le savoir, on fabrique un **faux QI** qui dépend vraiment de 5 indices HRV, avec une force connue : il explique 0 %, 5 %, 10 %, 20 % ou 30 % de la variance du QI (le R² « planté »), le reste étant du bruit. On lance ensuite exactement la même méthode (Elastic Net, Clinique + HRV, mêmes 189 enfants). On voit ainsi à partir de quelle force de lien notre méthode l'aurait détecté, et on situe notre vrai résultat sur cette échelle.

In [ ]:
nom_en = "Elastic Net · toutes · Clinique + HRV"
if nom_en in details_exp:
    retenues = [int(np.count_nonzero(d["QIT"]["coefs"])) for d in details_exp[nom_en] if "coefs" in d["QIT"]]
    part = np.mean(retenues) / len(JEUX["Clinique + HRV"])
    print(f"Elastic Net (Clinique + HRV, QIT) garde en moyenne {np.mean(retenues):.1f} variables sur "
          f"{len(JEUX['Clinique + HRV'])} par pli"
          + (" : presque aucune, il retombe sur la moyenne faute de lien fiable." if part < 0.1 else "."))

NIVEAUX = [0.0, 0.05, 0.10, 0.20, 0.30]
tirage = np.random.default_rng(GRAINE)
cols_signal = list(tirage.choice(cols_lin + cols_nl, size=5, replace=False))
z = X[cols_signal].apply(lambda s: s.fillna(s.median())).rank().sum(axis=1)
z = (z - z.mean()) / z.std()
bruit = pd.Series(tirage.standard_normal(len(X)), index=X.index)
qit = etiquettes["QIT"]
lignes = []
for r2 in NIVEAUX:
    faux_qit = qit.mean() + qit.std() * (np.sqrt(r2) * z + np.sqrt(1 - r2) * bruit)
    plis = plis_internes(faux_qit, pd.qcut(faux_qit, 5, labels=False))
    P = predictions_croisees(X, faux_qit, JEUX["Clinique + HRV"], faire_elastic_net, plis)
    lignes.append({"R² planté": r2, **gain_vs_moyenne(faux_qit, P, plis)})
simulation = pd.DataFrame(lignes).set_index("R² planté")
simulation["détecté"] = np.where(simulation["IC bas"] > 0, "oui", "non")
display(simulation[["MAE moyenne", "MAE modèle", "gain", "IC bas", "IC haut", "détecté"]].round(2))

reel = resume_equipe[(resume_equipe.target == "QIT") & (resume_equipe.model == nom_en)].iloc[0] \
    if nom_en in set(resume_equipe.model) else None
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.errorbar(simulation.index * 100, simulation["gain"],
            yerr=[simulation["gain"] - simulation["IC bas"], simulation["IC haut"] - simulation["gain"]],
            fmt="o-", capsize=3, label="faux QI : gain de MAE [IC 95 %]")
if reel is not None:
    ax.axhspan(reel.gain_ci_low, reel.gain_ci_high, color="tab:red", alpha=0.15)
    ax.axhline(reel.MAE_gain_vs_baseline, color="tab:red", lw=2, label=f"vrai QIT : {reel.MAE_gain_vs_baseline:+.2f} point")
ax.axhline(0, color="k", lw=0.8)
ax.set(xlabel="part de la variance du QI expliquée par la HRV dans le faux QI (%)",
       ylabel="gain de MAE sur la moyenne (points)", title="À partir de quand la méthode voit-elle un lien ?")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

detectes = simulation.index[simulation["IC bas"] > 0]
SEUIL_DETECTION = detectes.min() if len(detectes) else None
print((f"Avec ces {len(X)} enfants, la méthode détecte un lien dès qu'il explique environ "
       f"{SEUIL_DETECTION:.0%} de la variance du QI." if SEUIL_DETECTION is not None else
       "Aucun niveau simulé n'est détecté avec cette répétition : relancer avec RAPIDE = False.")
      + (" Notre vrai gain n'est pas démontré : si la HRV néonatale explique une part du QI à 3 ans, "
         "elle est plus petite que ce seuil." if reel is not None and reel.gain_ci_low <= 0 else ""))

# 5. Contrôle positif : la HRV retrouve-t-elle la prématurité ?

Un résultat nul peut avoir deux causes très différentes : **il n'y a pas de lien**, ou **notre méthode (ou nos données) est incapable d'en voir un**. Le contrôle positif sert à les distinguer : on pose à la même méthode une question dont on **connaît** la réponse.

On sait que la HRV d'un nouveau-né dépend fortement de sa maturité : un enfant né prématurément a un système nerveux autonome moins mûr, donc une HRV différente. On demande donc aux seuls indices HRV de deviner si l'enfant est prématuré.

- Si la HRV retrouve bien la prématurité (AUC nettement au-dessus de 0,5), **les données HRV contiennent de l'information physiologique et la méthode sait l'extraire**. L'absence de lien avec le QI devient alors un vrai résultat, pas un problème de données ou de code.
- Si elle n'y arrive pas, il faut d'abord se méfier des données (qualité des enregistrements, définition de la prématurité dans le fichier).

L'**AUC** se lit ainsi : 0,5 = le modèle ne fait pas mieux qu'une pièce de monnaie ; 1 = il sépare parfaitement les deux groupes ; au-dessus de 0,7, on parle en général d'une bonne discrimination.

In [ ]:
controle = None
col_prema = "clin|Prématuré"
if col_prema not in X or X[col_prema].nunique() < 2:
    print("Prématurité indisponible : pas de contrôle positif.")
else:
    renseigne = X[col_prema].dropna()
    prema = (renseigne == renseigne.max()).astype(int)          # 1 = prématuré (codage 0/1 du fichier)
    print(f"{prema.sum()} enfants prématurés sur {len(prema)} renseignés")
    if prema.sum() < 10 or (1 - prema).sum() < 10:
        print("Moins de 10 enfants dans un des deux groupes : contrôle trop fragile pour conclure.")
    else:
        plis_prema = plis_internes(prema, prema)
        P_prema = predictions_croisees(X, prema, cols_lin + cols_nl, faire_classifieur, plis_prema, proba=True)
        controle = resume_classification(prema, P_prema)
        display(pd.DataFrame([controle], index=["HRV seule → prématurité"]).round(2))
        print("Verdict :", "la HRV retrouve bien la prématurité : les données et la méthode sont capables de voir un lien."
              if controle["AUC IC bas"] > 0.5 else
              "la HRV ne retrouve pas la prématurité de façon démontrée : se méfier d'abord des données.")

# 6. Classification : repérer les enfants avec un QIT < 90

Prédire le QI au point près est peut-être trop demander. Une question plus modeste, et plus utile en clinique : **peut-on repérer les enfants dont le QIT sera inférieur à 90** (environ 1 enfant sur 4), pour les suivre de plus près ? Un signal limité aux QI bas pourrait passer inaperçu dans la régression.

**Le piège du déséquilibre.** Avec environ 78 % d'enfants au-dessus de 90, un « modèle » qui répond toujours « ≥ 90 » a 78 % d'exactitude, sans rien savoir. L'exactitude seule est donc trompeuse. On regarde plutôt :
- l'**AUC** : 0,5 = hasard, 1 = parfait ;
- la **sensibilité** (part des enfants < 90 repérés) et la **spécificité** (part des enfants ≥ 90 correctement laissés de côté) ;
- l'**exactitude équilibrée** (moyenne des deux : 0,5 = hasard) ;
- le **kappa de Cohen** (accord au-delà du hasard : 0 = hasard, 1 = parfait).

Le modèle est une régression logistique pénalisée, les deux classes comptant autant (`class_weight="balanced"`). Il est évalué sur les mêmes plis que la régression. La ligne « toujours ≥ 90 » montre le piège de l'exactitude.

In [ ]:
bas = (etiquettes["QIT"] < SEUIL_QIT).astype(int)
print(f"{bas.sum()} enfants sur {len(bas)} ({bas.mean():.0%}) ont un QIT < {SEUIL_QIT}")
classification = {f"toujours « ≥ {SEUIL_QIT} »": {"AUC": 0.5, "AUC IC bas": 0.5, "AUC IC haut": 0.5,
                                                  "sensibilité": 0.0, "spécificité": 1.0,
                                                  "exactitude": 1 - bas.mean(), "exactitude équilibrée": 0.5, "kappa": 0.0}}
for jeu in ["Clinique seul", "Clinique + HRV"]:
    if JEUX[jeu]:
        P = predictions_croisees(X, bas, JEUX[jeu], faire_classifieur, taches_plis(), proba=True)
        classification[f"Logistique · {jeu}"] = resume_classification(bas, P)
classification = pd.DataFrame(classification).T
display(classification[["AUC", "AUC IC bas", "AUC IC haut", "sensibilité", "spécificité", "exactitude",
                        "exactitude équilibrée", "kappa"]].round(2))

# 7. Hypothèses : HRV plus tardive et trajectoires

La HRV de la naissance ne prédit pas le QI à 3 ans. Deux hypothèses à tester avant de conclure que la HRV n'y est pour rien :

- **H1 : la HRV mesurée plus tard prédit mieux.** À 6, 12, 18 ou 24 mois, le système nerveux autonome est plus mûr et plus stable, et la mesure est plus proche de l'évaluation à 3 ans.
- **H2 : c'est l'évolution qui compte, pas la valeur à un âge donné.** Un enfant dont la HRV mûrit vite entre la naissance et 12 (ou 24) mois pourrait avoir un meilleur développement. Les variables sont alors les différences d'indices HRV entre les deux âges.

**Test** : pour chaque analyse, Elastic Net avec les covariables cliniques et les indices HRV, comparé au QIT moyen, en validation croisée sur les enfants qui ont les enregistrements nécessaires. Une hypothèse est soutenue si l'intervalle de confiance du gain est au-dessus de 0.

**Précautions** : les enfants diffèrent d'une analyse à l'autre (tous n'ont pas tous les enregistrements), donc les résultats ne se comparent pas au point près. Et avec 7 tests, un résultat positif isolé peut être dû au hasard : il faudrait le confirmer sur d'autres données avant d'y croire.

In [ ]:
lignes, jeux_passages = [], {}
for passage in PASSAGES:
    d, _ = construire_donnees(passage)
    jeux_passages[passage] = d
    colonnes = [c for c in d.columns if c.startswith(("lin|", "nl|", "clin|"))]
    if len(d) < 50 or not colonnes:
        print(f"{passage} : seulement {len(d)} enfants, analyse ignorée")
        continue
    plis = plis_internes(d["QIT"], pd.qcut(d["QIT"], 5, labels=False, duplicates="drop"))
    P = predictions_croisees(d, d["QIT"], colonnes, faire_elastic_net, plis)
    lignes.append({"analyse": f"H1 · HRV à {passage}", "variables": len(colonnes), **gain_vs_moyenne(d["QIT"], P, plis)})

for avant, apres in TRAJECTOIRES:
    d1, d2 = jeux_passages[avant], jeux_passages[apres]
    enfants = d1.index.intersection(d2.index)
    communes = [c for c in d1.columns if c.startswith(("lin|", "nl|")) and c in d2.columns]
    if len(enfants) < 50 or not communes:
        print(f"{avant} → {apres} : seulement {len(enfants)} enfants, analyse ignorée")
        continue
    evolution = (d2.loc[enfants, communes] - d1.loc[enfants, communes]).add_prefix("evol|")
    d = evolution.join(d1.loc[enfants, [c for c in d1.columns if c.startswith("clin|")] + ["QIT"]])
    colonnes = [c for c in d.columns if c != "QIT"]
    plis = plis_internes(d["QIT"], pd.qcut(d["QIT"], 5, labels=False, duplicates="drop"))
    P = predictions_croisees(d, d["QIT"], colonnes, faire_elastic_net, plis)
    lignes.append({"analyse": f"H2 · évolution {avant} → {apres}", "variables": len(colonnes),
                   **gain_vs_moyenne(d["QIT"], P, plis)})

hypotheses = pd.DataFrame(lignes).set_index("analyse")
hypotheses["soutenue"] = np.where(hypotheses["IC bas"] > 0, "oui", "non")
display(hypotheses[["enfants", "variables", "MAE moyenne", "MAE modèle", "gain", "IC bas", "IC haut", "soutenue"]].round(2))

fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(hypotheses) + 1.3))
y_pos = np.arange(len(hypotheses))[::-1]
ax.errorbar(hypotheses["gain"], y_pos, xerr=[hypotheses["gain"] - hypotheses["IC bas"], hypotheses["IC haut"] - hypotheses["gain"]],
            fmt="o", capsize=3)
ax.axvline(0, color="k", lw=0.8)
ax.set(yticks=y_pos, yticklabels=hypotheses.index, xlabel="gain de MAE sur le QIT moyen (points) [IC 95 %] ; à droite de 0 = mieux")
plt.tight_layout()
plt.show()

# 8. Prédiction conforme (QIT)

Un QIT prédit seul ne dit rien de sa marge d'erreur. La **prédiction conforme** donne un intervalle [prédiction − q ; prédiction + q] qui contient le vrai QIT avec une probabilité d'au moins 90 %, quel que soit le modèle : q est mesuré sur les erreurs du modèle chez des enfants qu'il n'a pas vus. Si l'intervalle n'est pas plus étroit que celui construit autour du simple QIT moyen, le modèle n'apporte rien pour un enfant donné. **Graphique individuel : à ne pas partager.**

In [ ]:
def quantile_conforme(erreurs, alpha=ALPHA):
    n = len(erreurs)
    return np.quantile(erreurs, min(1.0, np.ceil((n + 1) * (1 - alpha)) / n), method="higher")


def intervalle_un_pli(exp, train_ids, test_ids):
    etiq = etiquettes[["QIT"]]
    erreurs = []
    for tr_i, va_i in KFold(K_PLIS, shuffle=True, random_state=0).split(train_ids):
        tr_ids, va_ids = [train_ids[i] for i in tr_i], [train_ids[i] for i in va_i]
        p, _ = ajuster_predire(exp, tr_ids, va_ids, etiq)
        erreurs.extend(np.abs(etiq.loc[va_ids, "QIT"].to_numpy() - p["QIT"]))
    p, _ = ajuster_predire(exp, train_ids, test_ids, etiq)
    y_train = etiq.loc[train_ids, "QIT"]
    return pd.DataFrame({"vrai": etiq.loc[test_ids, "QIT"].to_numpy(), "pred": p["QIT"],
                         "q": quantile_conforme(np.array(erreurs)), "moyenne": y_train.mean(),
                         "q_ref": quantile_conforme(np.abs(y_train - y_train.mean()).to_numpy())}, index=test_ids)


conforme = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(intervalle_un_pli)(exp_meilleur, tr, te)
                                             for _, _, tr, te in taches_plis([1])))
couverture = (np.abs(conforme.vrai - conforme.pred) <= conforme.q).mean()
couverture_ref = (np.abs(conforme.vrai - conforme.moyenne) <= conforme.q_ref).mean()
q_final = quantile_conforme(np.abs(residus))
display(pd.DataFrame({"couverture observée": [f"{couverture:.0%}", f"{couverture_ref:.0%}"],
                      "demi-largeur (points de QIT)": [f"± {conforme.q.mean():.1f}", f"± {conforme.q_ref.mean():.1f}"]},
                     index=[MEILLEUR, "autour du QIT moyen"]))
print(f"Pour un nouvel enfant : QIT prédit ± {q_final:.1f} points, avec une probabilité d'au moins {1 - ALPHA:.0%} "
      "de contenir son vrai QIT.")

tri = conforme.sort_values("pred")
dedans = np.abs(tri.vrai - tri.pred) <= tri.q
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.vlines(range(len(tri)), tri.pred - tri.q, tri.pred + tri.q, color="lightsteelblue", lw=2)
ax.scatter(range(len(tri)), tri.vrai, s=10, c=np.where(dedans, "black", "tab:red"), zorder=3)
ax.axhline(SEUIL_QIT, color="grey", ls=":", lw=1)
ax.set(xlabel="enfants triés par QIT prédit", ylabel="QIT", title=f"Intervalles à {1 - ALPHA:.0%} (rouge : vrai QIT en dehors)")
plt.tight_layout()
plt.show()

# 9. Explication des variables

L'importance d'une variable se mesure ainsi : on mélange ses valeurs entre les enfants de test et on regarde de combien la performance baisse. Si elle baisse beaucoup, le modèle s'appuyait sur cette variable.

- **Pour le QI**, aucun modèle ne fait mieux que la moyenne (section 4) : l'importance de ses variables ne mesurerait que du bruit, et Elastic Net met presque tous ses coefficients à zéro (section 4.5). La réponse honnête à « quelle variable compte pour le QI ? » est donc : **aucune, avec ces données**.
- **Pour la prématurité** (contrôle positif, section 5), la HRV a un vrai signal : c'est là que l'explication a un sens. Le graphique montre les indices HRV qui permettent de reconnaître un enfant prématuré.

Une variable importante est **associée** au résultat dans cette cohorte ; ce n'est pas une **cause**. « Que faire pour un enfant prédit sous 90 ? » est une question causale : un modèle prédictif peut au mieux aider à repérer des enfants à suivre, et les interventions relèvent d'études cliniques.

In [ ]:
if controle is None:
    print("Pas de contrôle positif exploitable : pas d'explication à montrer.")
else:
    colonnes = cols_lin + cols_nl

    def importance_un_pli(tr, te):
        warnings.filterwarnings("ignore")
        pipe = faire_classifieur(colonnes).fit(X.loc[tr, colonnes], prema.loc[tr])
        r = permutation_importance(pipe, X.loc[te, colonnes], prema.loc[te], scoring="roc_auc", n_repeats=5, random_state=0)
        return pd.Series(r.importances_mean, index=colonnes)

    importances = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(importance_un_pli)(tr, te)
                                                    for r_, _, tr, te in plis_prema if r_ == 1), axis=1)
    haut = importances.mean(axis=1).sort_values(ascending=False).head(10)[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 4))
    ax.barh(haut.index, haut, color="tab:blue")
    ax.axvline(0, color="k", lw=0.8)
    ax.set(xlabel="baisse de l'AUC quand l'indice est mélangé", title="Indices HRV qui reconnaissent la prématurité")
    plt.tight_layout()
    plt.show()

# 10. Résumé pour le tuteur

Généré à partir des résultats ci-dessus (chiffres agrégés seulement) et enregistré dans `results/SUMMARY.md`.

In [ ]:
def ligne_resultat(cible):
    m = MEILLEURS[cible]
    c = comparaison_modeles[(comparaison_modeles.target == cible) & (comparaison_modeles.model == m)].iloc[0]
    ref = comparaison_modeles[(comparaison_modeles.target == cible) & (comparaison_modeles.model == REFERENCE)].iloc[0]
    e = resume_equipe[(resume_equipe.target == cible) & (resume_equipe.model == m)].iloc[0]
    return (f"| {cible} | {m} | {c.mae_mean:.2f} ± {c.mae_std:.2f} | {c.r2_mean:.3f} | {ref.mae_mean:.2f} | "
            f"{e.MAE_gain_vs_baseline:+.2f} {ic(e.gain_ci_low, e.gain_ci_high)} ({'démontré' if e.gain_ci_low > 0 else 'non démontré'}) |")


mae_qit = par_pli[par_pli.target == "QIT"].groupby("model")["MAE"].mean()
seul, avec = "Elastic Net · toutes · Clinique seul", "Elastic Net · toutes · Clinique + HRV"
demontres = [c for c in CIBLES
             if resume_equipe[(resume_equipe.target == c) & (resume_equipe.model == MEILLEURS[c])].gain_ci_low.iloc[0] > 0]
meilleure_classif = classification.drop(index=f"toujours « ≥ {SEUIL_QIT} »")["AUC"].idxmax()
c_classif = classification.loc[meilleure_classif]

SUMMARY = "\n".join([
    "# Prédiction du QI à 3 ans à partir de la HRV néonatale : résumé",
    f"*Groupe A · Jade · généré le {date.today():%d/%m/%Y} par `01_prediction_qit.ipynb`"
    + (" · mode rapide (1 répétition), chiffres provisoires*" if RAPIDE else f" · {T_REPETITIONS} × {K_PLIS} plis*"),
    "",
    "## Conclusion",
    ("**Aucun modèle ne fait mieux que prédire le score moyen, sur aucune cible.** " if not demontres else
     f"**Gain démontré par rapport au score moyen pour : {', '.join(demontres)}.** ")
    + (f"La HRV retrouve pourtant bien la prématurité (AUC {controle['AUC']:.2f}) : les données et la méthode savent "
       "voir un lien quand il existe." if controle is not None and controle["AUC IC bas"] > 0.5 else ""),
    "",
    "## Données et méthode",
    f"- {len(X)} enfants de la cohorte AuBE (enregistrement à la naissance, scores WPPSI-III complets) ; "
    f"{len(cols_lin)} indices HRV linéaires (périodes : {', '.join(sorted({periode(c) for c in cols_lin}))}), "
    f"{len(cols_nl)} non linéaires (périodes : {', '.join(sorted({periode(c) for c in cols_nl}))}), "
    f"{len(cols_clin)} covariables cliniques.",
    f"- Modèles : {', '.join(sorted({e[0] for e in EXPERIENCES}))} ; réduction des variables dans chaque pli ; "
    f"évaluation sur les plis communs de l'équipe ({T_REPETITIONS} × {K_PLIS}, `evaluate_model.py`).",
    "",
    "## Régression (meilleur modèle par cible)",
    "| Cible | Modèle | MAE ± écart-type | R² | MAE de la moyenne | Gain vs moyenne [IC 95 %] |",
    "|---|---|---|---|---|---|",
    *[ligne_resultat(c) for c in CIBLES],
    "",
    *([f"- Ajouter la HRV au modèle clinique (Elastic Net, QIT) : la MAE passe de {mae_qit[seul]:.2f} à {mae_qit[avec]:.2f} "
       f"({'mieux' if mae_qit[avec] < mae_qit[seul] else 'moins bien'})."] if seul in mae_qit and avec in mae_qit else []),
    f"- Signal dans les variables : la plus forte corrélation avec le QIT vaut {rho.iloc[0]:.2f}, contre {rho_hasard:.2f} "
    f"en moyenne par hasard (jusqu'à {seuil_hasard:.2f})"
    + (" : une corrélation dépasse le hasard." if SIGNAL_UNIVARIE else " : rien ne dépasse le hasard, pas de lien à apprendre."),
    *([f"- Limite de détection (simulation) : avec ces enfants, la méthode voit un lien dès qu'il explique environ "
       f"{SEUIL_DETECTION:.0%} de la variance du QI"
       + ("." if demontres else " ; si la HRV en explique une part, elle est plus petite.")]
      if SEUIL_DETECTION is not None else []),
    f"- Intervalle de prédiction à {1 - ALPHA:.0%} (prédiction conforme) : ± {q_final:.1f} points, contre "
    f"± {conforme.q_ref.mean():.1f} autour du QIT moyen.",
    "",
    "## Contrôle positif, classification, hypothèses",
    (f"- Contrôle positif : HRV seule → prématurité, AUC {controle['AUC']:.2f} "
     f"{ic(controle['AUC IC bas'], controle['AUC IC haut'])}." if controle is not None else
     "- Contrôle positif : prématurité indisponible ou trop rare."),
    f"- Classification QIT < {SEUIL_QIT} ({bas.sum()} enfants) : meilleure AUC {c_classif['AUC']:.2f} "
    f"{ic(c_classif['AUC IC bas'], c_classif['AUC IC haut'])} ({meilleure_classif}), exactitude équilibrée "
    f"{c_classif['exactitude équilibrée']:.2f}, kappa {c_classif['kappa']:.2f}.",
    *[f"- {a} ({int(h.enfants)} enfants) : gain {h.gain:+.2f} {ic(h['IC bas'], h['IC haut'])} "
      f"({'soutenue' if h['IC bas'] > 0 else 'non soutenue'})." for a, h in hypotheses.iterrows()],
    f"- Avec {len(hypotheses)} hypothèses testées, une hypothèse soutenue isolée peut être due au hasard : à confirmer.",
    "",
    "## Limites",
    "- Environ 190 enfants ; pas de cohorte externe ; le QI à 3 ans dépend de nombreux facteurs non mesurés ici "
    "(environnement familial, niveau socio-économique).",
    "- Une association n'est pas une cause : un modèle prédictif ne dit pas comment améliorer le QI d'un enfant.",
    "",
    "## Suite",
    "- Variables tirées des données brutes (fichiers RR, signal ECG) : travail du groupe B.",
    "- Confirmer sur une autre cohorte toute hypothèse soutenue ci-dessus.",
])
(DOSSIER_RESULTATS / "SUMMARY.md").write_text(SUMMARY + "\n", encoding="utf-8")
display(Markdown(SUMMARY))

# 11. Avant de committer

Le plan demande de vérifier qu'aucune donnée de santé n'est suivie par git. `results/` ne contient que des chiffres agrégés ; `data/` reste local.

In [ ]:
suivis = subprocess.run(["git", "ls-files"], cwd=RACINE, capture_output=True, text=True).stdout.splitlines()
suspects = [f for f in suivis if re.search(r"\.(csv|xlsx?|mat|edf|zip|pkl|joblib)$", f, re.I)]
print("Fichiers de données suivis par git :", suspects or "aucun")
print("Les sorties de ce notebook sont vidées au commit par nbstripout ; sans ce filtre, faire « Restart & Clear Output ».")